# Point-in-Time Check — *what did the algorithm say, and what happened next?*

Pick a stock and a **date in the past**. This notebook runs the screening funnel using *only* the data available on that date (no look-ahead), shows you the gate-by-gate verdict, and then reveals what the stock actually did over the following weeks — so you can judge whether the signal was right.

Edit the **Parameters** cell, then *Run All*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from breakout.config import Settings
from breakout.data.loader import load_or_fetch
from breakout.screen.funnel import screen_symbol

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True

## 1. Parameters

`AS_OF` is the day you 'stand on'. The funnel sees bars up to and including that day; `FORWARD_DAYS` is how far into its future we then look.

In [ ]:
TICKER       = 'SNDK'
BENCHMARK    = 'SPY'
AS_OF        = '2025-05-15'   # the date to evaluate the funnel on
FORWARD_DAYS = 40             # trading days to look ahead
EQUITY       = 100_000.0
FETCH_DAYS   = 730

## 2. Load bars

In [ ]:
df    = load_or_fetch(TICKER, lookback_days=FETCH_DAYS)
bench = load_or_fetch(BENCHMARK, lookback_days=FETCH_DAYS)
as_of = pd.Timestamp(AS_OF)
# snap to the last trading day on or before AS_OF
matching_dates = df.index[df.index <= as_of]
if matching_dates.empty:
    raise ValueError(f"No trading data available for {TICKER} on or before {AS_OF}. "
                     f"Data range: {df.index.min().date()} to {df.index.max().date()}. "
                     f"Please adjust AS_OF or FETCH_DAYS parameters.")
as_of = matching_dates[-1]
print(f'Evaluating {TICKER} as of {as_of.date()}')

## 3. What the funnel saw on that day

Gates (L0–L4) must all PASS for a signal; scores (L3/L5/L6) rank survivors.

In [ ]:
settings = Settings.load()
hist = df.loc[:as_of]                 # point-in-time: no future bars
bench_hist = bench.loc[:as_of]
cand = screen_symbol(TICKER, hist, settings, benchmark_df=bench_hist,
                     account_equity=EQUITY)

for lr in cand.layers:
    if lr.kind == 'gate':
        print(f'  [{"PASS" if lr.passed else "FAIL"}] {lr.name}')
    else:
        print(f'  [score={lr.score:.3f}] {lr.name}')
print()
if cand.passed_gates:
    r = cand.risk
    print(f'SIGNAL — composite {cand.composite:.3f}' + ('  (ATH breakout)' if cand.is_ath_breakout else ''))
    print(f'  entry  ~{r.entry:.2f}   stop {r.stop:.2f}   target {r.first_target:.2f} ({r.r_multiple_to_target:.1f}R)')
    print(f'  size   {r.shares} shares   risk ${r.risk_dollars:,.0f}')
else:
    print(f'NO SIGNAL — rejected at {cand.rejected_at}')

## 4. What happened next

The shaded region is the future the funnel could **not** see. If there was a signal, the entry, stop, and target levels are drawn so you can see whether price reached the target or hit the stop first.

In [ ]:
fwd = df.loc[as_of:].head(FORWARD_DAYS + 1)
past = df.loc[:as_of].tail(60)
fig, ax = plt.subplots()
ax.plot(past.index, past['close'], color='#444', lw=1.2, label='past (visible)')
ax.plot(fwd.index, fwd['close'], color='#2563eb', lw=1.6, label='future (hidden)')
ax.axvspan(as_of, fwd.index[-1], color='#2563eb', alpha=.05)
ax.axvline(as_of, color='#111', ls=':', lw=1)
if cand.passed_gates:
    r = cand.risk
    ax.axhline(r.entry, color='#2563eb', ls='-', lw=.8, alpha=.6, label='entry')
    ax.axhline(r.stop, color='#dc2626', ls='--', lw=.9, label='stop')
    ax.axhline(r.first_target, color='#16a34a', ls='--', lw=.9, label='target')
ax.set_title(f'{TICKER} — funnel verdict on {as_of.date()} and the {FORWARD_DAYS}-day future')
ax.legend(loc='upper left')
plt.show()

In [ ]:
# Did the future reach the target or the stop first? (only meaningful if there was a signal)
if cand.passed_gates and len(fwd) > 1:
    r = cand.risk
    after = fwd.iloc[1:]   # bars strictly after AS_OF
    hit_target = after.index[after['high'] >= r.first_target]
    hit_stop   = after.index[after['low']  <= r.stop]
    t_target = hit_target[0] if len(hit_target) else None
    t_stop   = hit_stop[0]   if len(hit_stop)   else None
    fwd_ret = (after['close'].iloc[-1] / r.entry - 1) * 100
    if t_target and (t_stop is None or t_target <= t_stop):
        print(f'TARGET hit first on {t_target.date()}  (+{r.r_multiple_to_target:.1f}R)')
    elif t_stop:
        print(f'STOP hit first on {t_stop.date()}  (about -1R)')
    else:
        print('Neither target nor stop hit within the window.')
    print(f'Buy-at-entry return over {FORWARD_DAYS} days: {fwd_ret:+.2f}%')
else:
    print('No signal on this date — nothing to track forward.')

---
Scan a few dates around a known breakout to build intuition for what the gates reward. For a full multi-trade run over a whole window, use **01_backtest_explorer.ipynb**.